# 第 23 课 · SDPA、FlashAttention 与推理性能测量

**从“公式正确”走向“知道实际运行了什么，以及怎样公平测量”。**

上一课已经实现 MHA/GQA/MQA 和紧凑 KV cache。本课不重训模型、不比较生成质量；
我们固定参数或 Q/K/V，把手写注意力替换为 PyTorch SDPA，再核对数学结果、执行后端、延迟和显存。

| 你将学会 | 可验收的作品 |
|---|---|
| 区分接口、算法和具体内核 | 后端探测报告与 profiler 算子记录 |
| 在线 softmax 的全局归一化 | 手写分块实现，与一次性计算核对 |
| 正确处理 prefill/decode/chunk | 没有未来泄漏的 SDPA 调用 |
| 正确计时与记录显存 | 同输入、多轮、预热、CUDA 同步的测量报告 |
| 不夸大性能结论 | 区分 attention core、整模型和服务端指标 |

**材料独立性：** 本 Notebook 不读取同名 `.py`，也不依赖前几课源码或检查点。配套脚本同样可独立运行。
本课只依赖 PyTorch 和标准库，不安装第三方 `flash-attn`，不下载模型或语料。

先完成前半部分的数学与接口实验，再完成基准与练习。运行全部单元不代表完成练习。

## 0. 环境与复现约定

沿用已经能运行前几课的环境。先不要为追求 GPU 重新安装整个环境。
本课使用 `F.scaled_dot_product_attention`、`torch.nn.attention.sdpa_kernel` 和 `torch.profiler`。
PyTorch 的后端支持会随版本、设备和输入变化，因此程序保存实际环境，并逐项探测。

配套文件的发布验证环境为 **Python 3.13.5 / PyTorch 2.10.0+cpu**；不能据此保证所有版本、所有 GPU 路径。
Notebook 中的默认设置是 CPU/FP32。已有可用 CUDA 后，可把下面的 `DEVICE_NAME` 改为 `"cuda"`，
`DTYPE_NAME` 改为 `"float16"`；这与“用 CPU 模拟显卡”完全不同。

在终端检查：
```bash
python -c "import torch; print(torch.__version__); print(torch.cuda.is_available())"
```
需要安装 GPU 构建时按 [PyTorch 官方安装页面](https://pytorch.org/get-started/locally/) 选择，
不盲目照抄一个 CUDA 版本号。

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
# %% 01. 环境与约定
"""第23课：SDPA、FlashAttention 与推理性能测量。

python lesson_23_sdpa_flashattention_and_benchmarking.py
python lesson_23_sdpa_flashattention_and_benchmarking.py --checks-only
python lesson_23_sdpa_flashattention_and_benchmarking.py --device auto --dtype auto

仅依赖 PyTorch 与标准库，不训练、不联网、不读取上一课检查点。
导入不运行实验、不写文件、不更改线程数。默认 CPU/FP32、单线程。
本课测 attention core，不是整模型、服务 TTFT、逐 token 延迟或 tokens/s。
计时不包含 QKV 投影、RoPE、缓存追加、mask 构造、校验和 profiler。
低精度手写基线把中间计算提升到 FP32；不将其称为同内核公平竞赛。
强制后端不可用时记录原因，不悄悄退回 math 后冒称 FlashAttention。
无 CUDA 时显存测量为 null，绝不把 CPU 的结果当成 3060 实测。
"""
from __future__ import annotations
import argparse
from contextlib import nullcontext
from dataclasses import asdict, dataclass
import gc
import json
import math
from pathlib import Path
import platform
import statistics
import time
import warnings
from typing import Callable

import torch
from torch import nn
from torch.nn import functional as F

try:
    from torch.nn.attention import SDPBackend, sdpa_kernel
except ImportError:
    SDPBackend = None
    sdpa_kernel = None

SOURCES = {
    'sdpa': 'https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html',
    'backend': 'https://docs.pytorch.org/docs/stable/generated/torch.nn.attention.sdpa_kernel.html',
    'flashattention': 'https://arxiv.org/abs/2205.14135',
    'tutorial': 'https://docs.pytorch.org/tutorials/intermediate/scaled_dot_product_attention_tutorial.html',
    'profiler': 'https://docs.pytorch.org/docs/stable/profiler.html',
    'cuda_sync': 'https://docs.pytorch.org/docs/stable/generated/torch.cuda.synchronize.html',
    'memory': 'https://docs.pytorch.org/docs/stable/generated/torch.cuda.memory.max_memory_allocated.html',
    'cuda_memory': 'https://docs.pytorch.org/docs/stable/notes/cuda.html#memory-management',
    'numerics': 'https://docs.pytorch.org/docs/stable/notes/numerical_accuracy.html',
}
DTYPES = {'float32': torch.float32, 'float16': torch.float16,
          'bfloat16': torch.bfloat16, 'float64': torch.float64}

In [2]:
def positive_int(value: int, name: str) -> None:
    if type(value) is not int or value <= 0:
        raise ValueError(f'{name} 必须是正整数，不接受 bool')


def choose_device(name: str) -> torch.device:
    if name not in ('cpu', 'cuda', 'auto'):
        raise ValueError('device 只能是 cpu/cuda/auto')
    if name == 'auto':
        name = 'cuda' if torch.cuda.is_available() else 'cpu'
    if name == 'cuda' and not torch.cuda.is_available():
        raise RuntimeError('CUDA 不可用；先用 --device cpu，或检查本机 PyTorch 安装')
    return torch.device(name)


def choose_dtype(name: str, device: torch.device) -> torch.dtype:
    if name == 'auto':
        return torch.float16 if device.type == 'cuda' else torch.float32
    if name not in DTYPES:
        raise ValueError('不支持的 dtype')
    dtype = DTYPES[name]
    if device.type == 'cpu' and dtype not in (torch.float32, torch.float64):
        raise ValueError('本课 CPU 基准仅支持 float32/float64；不把 CPU 半精度当作 GPU 代理')
    if device.type == 'cuda' and dtype == torch.bfloat16 and not torch.cuda.is_bf16_supported():
        raise RuntimeError('当前 CUDA 环境未报告 BF16 支持；可改用 float16')
    return dtype


def sync(device: torch.device) -> None:
    if device.type == 'cuda':
        torch.cuda.synchronize(device)


def environment(device: torch.device, dtype: torch.dtype) -> dict:
    result = {'python': platform.python_version(), 'platform': platform.platform(),
              'torch': str(torch.__version__), 'torch_cuda_build': torch.version.cuda,
              'device': str(device), 'dtype': str(dtype),
              'cpu_threads': torch.get_num_threads(), 'cuda_available': torch.cuda.is_available(),
              'sdpa_kernel_api': sdpa_kernel is not None,
              'float32_matmul_precision': torch.get_float32_matmul_precision(),
              'gpu': None, 'gpu_total_memory_bytes': None,
              'note': 'enabled 标志不等于当前输入实际选择了该内核；以探测与 profiler 为准'}
    if device.type == 'cuda':
        p = torch.cuda.get_device_properties(device)
        result.update(gpu=p.name, gpu_total_memory_bytes=p.total_memory,
                      capability=list(torch.cuda.get_device_capability(device)),
                      flash_sdp_enabled=torch.backends.cuda.flash_sdp_enabled(),
                      matmul_allow_tf32=torch.backends.cuda.matmul.allow_tf32)
    return result

In [3]:
DEVICE_NAME = "cpu"       # 明确选择设备；改成 auto 时不可用会回退 CPU
DTYPE_NAME = "auto"       # CPU→float32，CUDA→float16
OUTPUT_DIR = Path("lesson_23_outputs")
THREADS = 1

torch.set_num_threads(THREADS)
DEVICE = choose_device(DEVICE_NAME)
DTYPE = choose_dtype(DTYPE_NAME, DEVICE)
print(json.dumps(environment(DEVICE, DTYPE), ensure_ascii=False, indent=2))

{
  "python": "3.13.5",
  "platform": "Linux-6.18.57-x86_64-with-glibc2.41",
  "torch": "2.10.0+cpu",
  "torch_cuda_build": null,
  "device": "cpu",
  "dtype": "torch.float32",
  "cpu_threads": 1,
  "cuda_available": false,
  "sdpa_kernel_api": true,
  "float32_matmul_precision": "highest",
  "gpu": null,
  "gpu_total_memory_bytes": null,
  "note": "enabled 标志不等于当前输入实际选择了该内核；以探测与 profiler 为准"
}


## 1. 三个层次：数学目标、Python 接口、执行内核

数学目标仍然是：
\[
O=\operatorname{softmax}(QK^\top/\sqrt d+M)V.
\]

**SDPA** 是 scaled dot-product attention 的缩写；`F.scaled_dot_product_attention` 是实现它的接口。
**FlashAttention** 是一类通过分块、融合和减少显存往返来计算注意力的算法与实现。
同一个 SDPA 调用可以根据设备、精度、形状、mask 和后端设置选择不同路径。

所以以下推断都不成立：
- “我调用 SDPA，所以一定在跑 CUDA FlashAttention。”
- “`flash_sdp_enabled()` 是 True，所以这一组输入实际用了 Flash。”
- “函数名里出现 flash，所以一定是 GPU 执行。”

我们会保留自动分派、强制 math 和 CUDA-only 后端的独立记录，再看 profiler 中的实际算子。

来源：[PyTorch SDPA](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html)、
[后端选择上下文](https://docs.pytorch.org/docs/stable/generated/torch.nn.attention.sdpa_kernel.html)。

## 2. 统一输入约定，先保证比较的是同一个问题

| 张量 | 形状 | 含义 |
|---|---|---|
| Q | `[B,Hq,L,d]` | 本次需要计算的 Query |
| K、V | `[B,Hkv,S,d]` | 可读取的历史与本次 Key/Value |
| 输出 | `[B,Hq,L,d]` | 每个 Query 汇总后的结果 |

本课要求 `Hq % Hkv == 0`，采用连续分组，`K` 和 `V` 同形；不把所有一般 SDPA 输入都纳入这份小练习。
布尔 mask 的 `True` 表示允许，不是屏蔽。手写参考和 SDPA 使用相同约定。
本课明确拒绝一整行完全屏蔽，不依赖不同实现对此的特殊处理。

**性能基准的前提：** 等长、无 padding 的请求；K/V 包含本次新增位置。
有 padding 或复杂位置布局时，需要重新设计和核对 mask，不能原样照搬。

In [4]:
def validate_qkv(q: torch.Tensor, k: torch.Tensor, v: torch.Tensor) -> None:
    for name, x in [('q', q), ('k', k), ('v', v)]:
        if (not isinstance(x, torch.Tensor) or x.ndim != 4 or min(x.shape) <= 0
                or x.dtype not in DTYPES.values()):
            raise ValueError(f'{name} 必须是非空浮点 [B,H,T,d]')
    if not (q.device == k.device == v.device and q.dtype == k.dtype == v.dtype):
        raise ValueError('Q/K/V 必须同设备、同 dtype')
    if q.shape[0] != k.shape[0] or k.shape != v.shape or q.shape[-1] != k.shape[-1]:
        raise ValueError('本课要求相同 batch、相同每头维度，K/V 完全同形')
    if q.shape[1] % k.shape[1]:
        raise ValueError('Hq 必须能被 Hkv 整除')


def validate_mask(mask: torch.Tensor | None, q: torch.Tensor, k: torch.Tensor,
                  causal: bool) -> None:
    if type(causal) is not bool:
        raise ValueError('is_causal 必须是 bool')
    if mask is not None:
        if causal:
            raise ValueError('本课不同时传入 mask 和 is_causal=True')
        if mask.dtype != torch.bool or mask.device != q.device:
            raise ValueError('本课 mask 必须为同设备 bool；True=允许')
        if mask.shape != (q.shape[-2], k.shape[-2]):
            raise ValueError('本课 mask 必须是 [L,S]，不接受隐式错误广播')
        if not bool(mask.any(dim=-1).all()):
            raise ValueError('本课拒绝全屏蔽行；不依赖不同后端对此的特殊行为')


def offset_mask(past_len: int, new_len: int, device='cpu') -> torch.Tensor:
    if type(past_len) is not int or past_len < 0:
        raise ValueError('past_len 必须是非负整数')
    positive_int(new_len, 'new_len')
    q_pos = past_len + torch.arange(new_len, device=device)
    k_pos = torch.arange(past_len + new_len, device=device)
    return k_pos[None, :] <= q_pos[:, None]


def attention_plan(q_len: int, kv_len: int, phase: str, device='cpu') -> tuple:
    """无 padding、历史完整有效、K/V 已包含本次新增位置时才适用。"""
    positive_int(q_len, 'q_len'); positive_int(kv_len, 'kv_len')
    if phase == 'prefill':
        if q_len != kv_len:
            raise ValueError('prefill 要求 L=S')
        return None, True
    if phase == 'decode':
        if q_len != 1:
            raise ValueError('decode 在本课专指一个新增 token')
        return None, False  # 所有 key 都是历史或当前自身，没有未来 key。
    if phase == 'chunk':
        if kv_len < q_len:
            raise ValueError('chunk 要求 S>=L')
        return offset_mask(kv_len - q_len, q_len, device), False
    raise ValueError('phase 必须是 prefill/decode/chunk')

In [5]:
print("prefill：", attention_plan(5, 5, "prefill"))
print("单 token decode：", attention_plan(1, 5, "decode"))
chunk_mask, chunk_causal = attention_plan(2, 5, "chunk")
print("chunk，历史3个+新增2个：")
print(chunk_mask)
print("is_causal：", chunk_causal)

prefill： (None, True)
单 token decode： (None, False)
chunk，历史3个+新增2个：
tensor([[ True,  True,  True,  True, False],
        [ True,  True,  True,  True,  True]])
is_causal： False


## 3. 把前几课的手写核心作为可读的参考

`manual_core` 使用分组维度 `[B,Hkv,G,L,S]`，不先复制 K/V。
但它仍然显式形成完整分数与权重。这一份大中间结果正是后面要理解的优化对象。

FP16/BF16 输入会把中间计算提升到 FP32；FP64 保留。它是数值清楚的教学参考，
不是为了击败融合内核而写的高性能实现。低精度时的类型转换也会计入它自己的耗时。

本课优化路径只返回输出，不要求返回整张注意力图；否则会把需要避免的大矩阵重新引入。

In [6]:
def manual_core(q, k, v, mask=None, is_causal=False, dropout_p=0.0):
    """显式物化 [B,Hkv,G,L,S] 分数；无校验的热路径，只在已检查输入上使用。

    FP16/BF16 输入先升 FP32，FP64 保留。没有显式复制 K/V。
    不返回权重，否则会给实际优化路径增加本不需要的输出。
    """
    original_dtype = q.dtype
    work = torch.float64 if q.dtype == torch.float64 else torch.float32
    q, k, v = q.to(work), k.to(work), v.to(work)
    B, Hq, L, d = q.shape
    Hkv, S = k.shape[1], k.shape[2]
    grouped_q = q.reshape(B, Hkv, Hq // Hkv, L, d)
    scores = (grouped_q @ k.unsqueeze(2).transpose(-2, -1)) / math.sqrt(d)
    if is_causal:
        mask = torch.ones(L, S, dtype=torch.bool, device=q.device).tril()
    if mask is not None:
        scores = scores.masked_fill(~mask, float('-inf'))
    weights = scores.softmax(dim=-1)
    if dropout_p:
        weights = F.dropout(weights, p=dropout_p, training=True)
    output = weights @ v.unsqueeze(2)
    return output.reshape(B, Hq, L, d).to(original_dtype)


def manual_attention(q, k, v, mask=None, is_causal=False):
    validate_qkv(q, k, v); validate_mask(mask, q, k, is_causal)
    return manual_core(q, k, v, mask, is_causal)

In [7]:
def make_qkv(B,Hq,Hkv,L,S,d,device,dtype,seed=2300):
    gen=torch.Generator(device='cpu').manual_seed(seed)
    return tuple(torch.randn(shape,generator=gen,dtype=torch.float64).to(device=device,dtype=dtype)
                 for shape in [(B,Hq,L,d),(B,Hkv,S,d),(B,Hkv,S,d)])


def error_stats(actual, expected) -> dict:
    a,b=actual.detach().double(),expected.detach().double()
    delta=(a-b).abs()
    return {'max_abs':float(delta.max()),
            'relative_l2':float(torch.linalg.vector_norm(a-b)/torch.linalg.vector_norm(b).clamp_min(1e-12))}


def tolerance(dtype):
    return {torch.float64:(1e-9,1e-10),torch.float32:(2e-4,2e-5),
            torch.float16:(1e-2,2e-3),torch.bfloat16:(4e-2,2e-2)}[dtype]


def require_close(a,b,dtype=None):
    rtol,atol=tolerance(dtype or a.dtype)
    torch.testing.assert_close(a.double(),b.double(),rtol=rtol,atol=atol)

In [8]:
q, k, v = make_qkv(1, 4, 2, 5, 5, 8, DEVICE, torch.float64)
manual = manual_attention(q, k, v, is_causal=True)
print("Q / K / V：", q.shape, k.shape, v.shape)
print("输出：", manual.shape)
print("第0位置只能读取自己，对应的两个Query头共享V：")
print(manual[0, :2, 0, :3])

Q / K / V： torch.Size([1, 4, 5, 8]) torch.Size([1, 2, 5, 8]) torch.Size([1, 2, 5, 8])
输出： torch.Size([1, 4, 5, 8])
第0位置只能读取自己，对应的两个Query头共享V：
tensor([[-0.1913,  1.0665,  0.3400],
        [-0.1913,  1.0665,  0.3400]], dtype=torch.float64)


## 4. 使用 SDPA：缩放、softmax、dropout 分别由谁负责？

SDPA 内部包含按每头维度缩放和 softmax，所以不要先对 Q/K 再除一遍 $\sqrt d$，
也不要把已经 softmax 的分数当作 Q/K 传进去。

GQA 使用 `enable_gqa=True`。本课不先把 KV 展开再称为“原生紧凑 GQA”。
原生支持会随后端变化，后面逐项探测；不可用就记录，不偷换成其他算法后声称成功。

**Dropout 必须显式控制。** 直接调用函数时，`dropout_p=0.1` 在 `eval()` 外壳下也不会自己变成0。
模块里应写 `self.dropout if self.training else 0.0`。

来源：[SDPA 参数与警告](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html)。

In [9]:
def backend_context(name: str):
    if name == 'auto':
        return nullcontext()
    if sdpa_kernel is None:
        raise RuntimeError('该 PyTorch 没有 torch.nn.attention.sdpa_kernel；请检查版本')
    mapping = {'math': 'MATH', 'flash': 'FLASH_ATTENTION',
               'efficient': 'EFFICIENT_ATTENTION', 'cudnn': 'CUDNN_ATTENTION'}
    if name not in mapping or not hasattr(SDPBackend, mapping[name]):
        raise ValueError(f'未知或当前版本未提供的后端：{name}')
    return sdpa_kernel(backends=[getattr(SDPBackend, mapping[name])])


def sdpa_raw(q, k, v, mask=None, is_causal=False, dropout_p=0.0):
    # 不二次乘 1/sqrt(d)；不先 softmax；不重复 K/V；不隐藏后端错误。
    kwargs = dict(attn_mask=mask, is_causal=is_causal, dropout_p=float(dropout_p))
    if q.shape[1] != k.shape[1]:
        kwargs['enable_gqa'] = True
    return F.scaled_dot_product_attention(q, k, v, **kwargs)


def sdpa_attention(q, k, v, mask=None, is_causal=False, *, backend='auto',
                   dropout_p=0.0):
    validate_qkv(q, k, v); validate_mask(mask, q, k, is_causal)
    if (isinstance(dropout_p, bool) or not isinstance(dropout_p, (float, int))
            or not math.isfinite(dropout_p) or not 0 <= dropout_p < 1):
        raise ValueError('dropout_p 必须在 [0,1) 内')
    with backend_context(backend):
        return sdpa_raw(q, k, v, mask, is_causal, dropout_p)

In [10]:
official = sdpa_attention(q, k, v, is_causal=True, backend="math")
print("手写 vs math：", error_stats(official, manual))
torch.testing.assert_close(official, manual, rtol=1e-9, atol=1e-10)

# 输出相同还不够：训练需要梯度也正确。
qa, ka, va = [t.clone().requires_grad_() for t in (q,k,v)]
qb, kb, vb = [t.clone().requires_grad_() for t in (q,k,v)]
loss_a = manual_attention(qa,ka,va,is_causal=True).square().mean()
loss_b = sdpa_attention(qb,kb,vb,is_causal=True,backend="math").square().mean()
ga = torch.autograd.grad(loss_a, (qa,ka,va))
gb = torch.autograd.grad(loss_b, (qb,kb,vb))
for name, a, b in zip(("Q","K","V"), ga, gb):
    print(name, "梯度最大差异：", (a-b).abs().max().item())

手写 vs math： {'max_abs': 4.440892098500626e-16, 'relative_l2': 1.0199156361693113e-16}
Q 梯度最大差异： 9.86623976961809e-18
K 梯度最大差异： 1.214306433183765e-17
V 梯度最大差异： 1.3877787807814457e-17


## 5. FlashAttention 的主要思路：减少搬运，不是删掉大部分 token

标准实现会形成 $L\times S$ 分数与概率矩阵。长序列时，写入和再次读取这些大中间结果有明显成本。
FlashAttention 用分块与融合控制数据在 GPU 显存和片上存储之间的移动，
避免完整注意力矩阵在显存中反复物化；训练反向还结合重计算。

**它不是本课所说的稀疏注意力、低秩近似或量化。** 在实数算术的意义上计算相同的 dense attention；
浮点实现允许有误差。方阵 dense attention 的主要乘加量仍随 $T^2d$ 增长，
不能把“中间显存更省”误写成“算术复杂度变成线性”。

一个 Q 向量需要所有允许 Key 的**全局** softmax 分母。
分块后如何保留这个分母，是接下来亲手验证的重点。

来源：[FlashAttention 原论文](https://arxiv.org/abs/2205.14135)。

## 6. 反例：每块单独 softmax，最后平均，为什么不行？

四个 logits 为 `[0,0,3,3]`，对应标量 Value 为 `[0,0,10,10]`。

一次性计算，后两个候选概率更高，输出约为 **9.525741**。
如果前两个和后两个分别 softmax，得到两个局部输出 `0`、`10`，再平均得到 `5`。
错误来自丢掉了两块之间的概率质量比例。

### 用三个摘要合并分块
对每个 Query 保存：
\[
m=\max_j s_j,\quad
\ell=\sum_j e^{s_j-m},\quad
u=\sum_j e^{s_j-m}v_j.
\]

新块最大值为 $m_b$，先更新 $m'=\max(m,m_b)$，再更新：
\[
\alpha=e^{m-m'},\qquad
\ell'=\alpha\ell+\sum_{j\in b}e^{s_j-m'},
\]
\[
u'=\alpha u+\sum_{j\in b}e^{s_j-m'}v_j.
\]

所有块完成后返回 $u/\ell$。新块提高最大值时，旧摘要必须重新缩放。
实现还要处理“当前块某行全被屏蔽”的情况，避免 `-inf - (-inf)`。

In [11]:
def merge_summary(m, z, u, block_scores, block_values):
    """把一个 key 块合入每个 query 的 (最大值, 指数和, 加权分子)。

    m,z: [L,1]；u: [L,Dv]；block_scores: [L,C]；block_values: [C,Dv]。
    允许某个块对某行全部被屏蔽；所有块结束后每行至少有一个有效 key。
    """
    block_max = block_scores.max(dim=-1, keepdim=True).values
    new_m = torch.maximum(m, block_max)
    safe_m = torch.where(torch.isfinite(new_m), new_m, torch.zeros_like(new_m))
    alpha = torch.exp(m - safe_m)  # m=-inf 时为0，避免 -inf - -inf。
    p = torch.exp(block_scores - safe_m)
    return new_m, alpha * z + p.sum(dim=-1, keepdim=True), alpha * u + p @ block_values


def streaming_attention(q, k, v, block_size=2, mask=None):
    """单个头的二维讲解实现，只用于小输入与验证，不计入性能竞赛。

    前向不先构造完整 LxS 分数，而逐块计算。Python 循环不控制片上 SRAM；
    普通 autograd 还会保存循环中的图，不能据此声称训练显存也线性。
    """
    positive_int(block_size, 'block_size')
    if (q.ndim != 2 or k.ndim != 2 or v.ndim != 2 or min(q.shape) <= 0
            or k.shape[0] == 0 or v.shape[1] == 0 or q.shape[1] != k.shape[1]
            or k.shape[0] != v.shape[0]):
        raise ValueError('需要非空 Q[L,d]、K[S,d]、V[S,dv]')
    if not (q.dtype == k.dtype == v.dtype and q.device == k.device == v.device
            and q.dtype in (torch.float32, torch.float64)):
        raise ValueError('在线教学实现要求同设备同 dtype 的 FP32/FP64')
    L, d = q.shape
    S = k.shape[0]
    if mask is not None:
        if (mask.shape != (L, S) or mask.dtype != torch.bool or mask.device != q.device
                or not bool(mask.any(-1).all())):
            raise ValueError('mask 必须为同设备 [L,S] bool，且没有全屏蔽行')
    m = q.new_full((L, 1), float('-inf'))
    z = q.new_zeros((L, 1))
    u = q.new_zeros((L, v.shape[-1]))
    for start in range(0, S, block_size):
        stop = min(start + block_size, S)
        scores = (q @ k[start:stop].T) / math.sqrt(d)
        if mask is not None:
            scores = scores.masked_fill(~mask[:, start:stop], float('-inf'))
        m, z, u = merge_summary(m, z, u, scores, v[start:stop])
    return u / z


def online_demo() -> dict:
    q = torch.ones(1, 1, dtype=torch.float64)
    k = torch.tensor([[0.0], [0.0], [3.0], [3.0]], dtype=torch.float64)
    v = torch.tensor([[0.0], [0.0], [10.0], [10.0]], dtype=torch.float64)
    expected = ((q @ k.T).softmax(-1) @ v).item()
    streamed = streaming_attention(q, k, v, block_size=2).item()
    local_mean = sum(((q @ k[s:s+2].T).softmax(-1) @ v[s:s+2]).item()
                     for s in (0, 2)) / 2
    return {'global_softmax_output': expected, 'online_output': streamed,
            'wrong_average_of_local_outputs': local_mean,
            'note': '不能分别对每块做 softmax 后简单平均；需要全局归一化摘要'}

In [12]:
online_result = online_demo()
print(json.dumps(online_result, ensure_ascii=False, indent=2))

q_one = torch.tensor([[1.,0.],[0.,1.]], dtype=torch.float64)
k_one = torch.tensor([[1.,0.],[0.,1.],[1.,1.]], dtype=torch.float64)
v_one = torch.arange(6,dtype=torch.float64).reshape(3,2)
mask_one = offset_mask(1,2)
full_scores = (q_one @ k_one.T / math.sqrt(2)).masked_fill(~mask_one, float("-inf"))
expected = full_scores.softmax(-1) @ v_one
for block_size in (1,2,7):
    actual = streaming_attention(q_one,k_one,v_one,block_size,mask_one)
    print("块长",block_size,"误差：",(actual-expected).abs().max().item())

{
  "global_softmax_output": 9.525741268224332,
  "online_output": 9.525741268224333,
  "wrong_average_of_local_outputs": 5.0,
  "note": "不能分别对每块做 softmax 后简单平均；需要全局归一化摘要"
}
块长 1 误差： 4.440892098500626e-16
块长 2 误差： 4.440892098500626e-16
块长 7 误差： 4.440892098500626e-16


### 边界说明：上面的循环不是“自己写好了 FlashAttention”

它只演示分块前向如何保持全局归一化。Python/Torch 运算没有显式控制片上 SRAM 的加载和复用，
也没有写融合 CUDA 内核或定制 backward。
普通 autograd 可能保留循环中的多份中间状态，所以不能仅凭这段代码声称训练显存也被优化了。
本课不会把它加入速度竞赛；真正的高效实现交给支持的 SDPA 后端。

## 7. 非方阵 `is_causal=True` 的坑：单 token decode 并非普通左上三角形

PyTorch SDPA 的 `is_causal=True` 在矩形输入上使用左上角对齐。
如果 Query 长度为1、K/V长度为4，它只允许读取第一个 Key，
并不知道这个 Query 实际位于历史末端。

无 padding、全部 K/V 都是历史或当前自身时，单 token decode 应允许读取全部 Key：
`attn_mask=None, is_causal=False`。多个新增 token 的 chunk 则显式构造带偏移的 mask。

下面的所有 logits 都为0，正确答案是四个 Value 的平均值 **2.5**；错误调用得到 **1.0**。

来源：[SDPA 的非方阵因果约定](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html)。

In [13]:
def rectangular_demo() -> dict:
    q = torch.zeros(1, 1, 1, 2, dtype=torch.float64)
    k = torch.zeros(1, 1, 4, 2, dtype=torch.float64)
    v = torch.tensor([1., 2., 3., 4.], dtype=torch.float64).reshape(1,1,4,1).expand(-1,-1,-1,2)
    good = sdpa_attention(q, k, v, backend='math')
    bad = sdpa_attention(q, k, v, is_causal=True, backend='math')
    return {'decode_correct': good[0,0,0,0].item(),
            'wrong_upper_left_causal': bad[0,0,0,0].item(),
            'chunk_past3_new2_mask': offset_mask(3,2).tolist()}

print(json.dumps(rectangular_demo(), ensure_ascii=False, indent=2))

{
  "decode_correct": 2.5,
  "wrong_upper_left_causal": 1.0,
  "chunk_past3_new2_mask": [
    [
      true,
      true,
      true,
      true,
      false
    ],
    [
      true,
      true,
      true,
      true,
      true
    ]
  ]
}


## 8. 接入投影、RoPE 与紧凑缓存：只换核心，不偷偷换模型

下面的小模块复用你已经学过的部件：Q/K/V 投影、相邻配对 RoPE、GQA、输出投影和动态缓存。
它不包含 FFN 或词表头，**不是重新训练一份 GPT**。

`backend="manual"` 与 `backend="math"` 使用同一组参数；缓存始终保存 `[B,Hkv,T,d]`，
其中 K 已旋转、V 未旋转。一个 chunk 只传入还未处理的新 token。

我们同时核对输出、投影梯度和各层缓存内容，不能只看随机生成的一句话是否一样。

In [14]:
def rotary(x, positions):
    """与前课相同的相邻坐标配对；本模块只做 FP32/FP64 集成验证。"""
    d = x.shape[-1]
    freq = 10000.0 ** (-torch.arange(0, d, 2, device=x.device, dtype=x.dtype) / d)
    angles = positions.to(x.dtype)[:, None] * freq[None, :]
    c, s = angles.cos()[None, None], angles.sin()[None, None]
    a, b = x[..., 0::2], x[..., 1::2]
    return torch.stack((a*c-b*s, a*s+b*c), dim=-1).flatten(-2)


class CoreSwapAttention(nn.Module):
    """无 padding 的 GQA+RoPE 小模块；不加载检查点，不冒充完整语言模型。"""
    def __init__(self, dim=32, heads=4, kv_heads=2, dropout=0.0):
        super().__init__()
        for name, value in [('dim',dim),('heads',heads),('kv_heads',kv_heads)]:
            positive_int(value, name)
        if dim % heads or heads % kv_heads or (dim//heads) % 2:
            raise ValueError('需 D%Hq=0、Hq%Hkv=0，且每头维度为偶数')
        if not isinstance(dropout, (int,float)) or isinstance(dropout,bool) or not 0<=dropout<1:
            raise ValueError('dropout 必须在 [0,1)')
        self.dim, self.heads, self.kv_heads = dim, heads, kv_heads
        self.d, self.dropout = dim // heads, float(dropout)
        self.q_proj = nn.Linear(dim, dim, bias=False)
        self.k_proj = nn.Linear(dim, kv_heads*self.d, bias=False)
        self.v_proj = nn.Linear(dim, kv_heads*self.d, bias=False)
        self.out_proj = nn.Linear(dim, dim, bias=False)

    def forward(self, x, *, cache=None, backend='auto'):
        if x.ndim != 3 or x.shape[-1] != self.dim or min(x.shape)<=0:
            raise ValueError('x 必须是非空 [B,T,D]')
        B, T, _ = x.shape
        past_len = 0
        if cache is not None:
            if self.training or torch.is_grad_enabled():
                raise ValueError('缓存路径要求 eval 且 no_grad/inference_mode')
            if not isinstance(cache, tuple) or len(cache)!=2:
                raise ValueError('cache 应为 (K,V)')
            old_k, old_v = cache
            if (old_k.ndim!=4 or old_k.shape!=old_v.shape or old_k.shape[:2]!=(B,self.kv_heads)
                    or old_k.shape[-1]!=self.d or old_k.shape[-2]==0
                    or old_k.dtype!=x.dtype or old_v.dtype!=x.dtype
                    or old_k.device!=x.device or old_v.device!=x.device):
                raise ValueError('缓存的 batch/头数/每头维度/device/dtype 不匹配')
            past_len=old_k.shape[-2]
        positions = torch.arange(past_len, past_len+T, device=x.device)
        q = self.q_proj(x).reshape(B,T,self.heads,self.d).transpose(1,2)
        k = self.k_proj(x).reshape(B,T,self.kv_heads,self.d).transpose(1,2)
        v = self.v_proj(x).reshape(B,T,self.kv_heads,self.d).transpose(1,2)
        q, k = rotary(q,positions), rotary(k,positions)
        if cache is not None:
            k, v = torch.cat((old_k,k),-2), torch.cat((old_v,v),-2)
        phase = 'prefill' if past_len==0 else ('decode' if T==1 else 'chunk')
        mask, causal = attention_plan(T,k.shape[-2],phase,x.device)
        p = self.dropout if self.training else 0.0
        if backend == 'manual':
            output=manual_core(q,k,v,mask,causal,p)
        else:
            output=sdpa_attention(q,k,v,mask,causal,backend=backend,dropout_p=p)
        merged=output.transpose(1,2).contiguous().reshape(B,T,self.dim)
        return self.out_proj(merged), (k,v)

In [15]:
def integration_demo(device=torch.device('cpu')) -> dict:
    with torch.random.fork_rng(devices=[] if device.type=='cpu' else [torch.cuda.current_device()]):
        torch.manual_seed(2301)
        model=CoreSwapAttention().to(device=device,dtype=torch.float64).eval()
        x=torch.randn(2,7,32,device=device,dtype=torch.float64)
    before={k:v.detach().clone() for k,v in model.state_dict().items()}
    with torch.inference_mode():
        ref, ref_cache=model(x,backend='manual')
        full, full_cache=model(x,backend='math')
        chunks=[]; cache=None; start=0
        for size in (2,1,4):
            y,cache=model(x[:,start:start+size],cache=cache,backend='math')
            chunks.append(y); start+=size
        chunked=torch.cat(chunks,dim=1)
    unchanged=all(torch.equal(v,before[k]) for k,v in model.state_dict().items())
    return {'dtype':'float64','manual_vs_sdpa_max_abs':float((ref-full).abs().max()),
            'full_vs_chunks_max_abs':float((full-chunked).abs().max()),
            'K_max_abs':float((ref_cache[0]-cache[0]).abs().max()),
            'V_max_abs':float((ref_cache[1]-cache[1]).abs().max()),
            'compact_cache_shape':list(cache[0].shape),'weights_unchanged':unchanged,
            'note':'同一模块的固定权重；随机输入检验计算机制，不评估语言质量'}

integration_result = integration_demo(DEVICE)
print(json.dumps(integration_result,ensure_ascii=False,indent=2))

{
  "dtype": "float64",
  "manual_vs_sdpa_max_abs": 2.498001805406602e-16,
  "full_vs_chunks_max_abs": 1.6653345369377348e-16,
  "K_max_abs": 3.3306690738754696e-16,
  "V_max_abs": 6.661338147750939e-16,
  "compact_cache_shape": [
    2,
    2,
    7,
    8
  ],
  "weights_unchanged": true,
  "note": "同一模块的固定权重；随机输入检验计算机制，不评估语言质量"
}


## 9. 强制后端探测：不支持就诚实记录

自动分派 `auto` 成功不代表强制 `flash` 也能成功。强制探测使用：
```python
with sdpa_kernel(backends=[SDPBackend.FLASH_ATTENTION]):
    output = F.scaled_dot_product_attention(q,k,v,...)
```
只启用目标后端，不把 math 放进候选列表偷偷回退。
探测会覆盖 MHA/GQA/MQA，以及 prefill、decode、chunk 三种形状。

状态含义：`ok`=该调用运行且通过数值检查；`skipped`=本课未测试（例如没有CUDA）；
`unsupported_or_error`=调用失败并保存原因；`correctness_failed`=运行了但数值核对未通过。
**后两者不能计作通过。**

3060 上的可用性取决于实际软件构建、dtype、head dimension、mask 和 GQA 支持；
本课不凭显卡名字承诺某个后端必定可用，也不要求安装独立 `flash-attn` 包。

来源：[后端上下文](https://docs.pytorch.org/docs/stable/generated/torch.nn.attention.sdpa_kernel.html)。

In [16]:
def probe_one(q,k,v,mask,causal,backend) -> dict:
    result={'backend_request':backend,'status':'not_run','warnings':[]}
    if q.device.type!='cuda' and backend in ('flash','efficient','cudnn'):
        return {**result,'status':'skipped',
                'reason':'此项专门探测 CUDA 后端；CPU 自动分派另行记录，不能当作 GPU 结果'}
    try:
        with warnings.catch_warnings(record=True) as caught, torch.inference_mode():
            warnings.simplefilter('always')
            with backend_context(backend):
                out=sdpa_raw(q,k,v,mask,causal)
                sync(q.device)
            result['warnings']=[str(w.message) for w in caught]
            ref=manual_attention(q.double(),k.double(),v.double(),mask,causal)
            require_close(out,ref,q.dtype)
        return {**result,'status':'ok',**error_stats(out,ref)}
    except AssertionError as exc:
        return {**result,'status':'correctness_failed','reason':str(exc)}
    except (RuntimeError,TypeError,ValueError,NotImplementedError) as exc:
        return {**result,'status':'unsupported_or_error','reason':f'{type(exc).__name__}: {exc}',
                'warnings':[str(w.message) for w in locals().get('caught',[])]}


def backend_probes(device,dtype) -> list[dict]:
    rows=[]
    for Hkv in (4,2,1):
        for phase,L,S in [('prefill',8,8),('decode',1,8),('chunk',3,8)]:
            q,k,v=make_qkv(1,4,Hkv,L,S,32,device,dtype)
            mask,causal=attention_plan(L,S,phase,device)
            for backend in ('auto','math','flash','efficient','cudnn'):
                rows.append({'phase':phase,'Hq':4,'Hkv':Hkv,'L':L,'S':S,
                             'dtype':str(dtype),**probe_one(q,k,v,mask,causal,backend)})
    return rows


def profile_dispatch(device,dtype,Hkv=4) -> dict:
    """Profiler 不参与延迟测量；记录实际算子名称，不固定猜测命名。"""
    q,k,v=make_qkv(1,4,Hkv,16,16,32,device,dtype)
    activities=[torch.profiler.ProfilerActivity.CPU]
    if device.type=='cuda':
        activities.append(torch.profiler.ProfilerActivity.CUDA)
    try:
        with torch.inference_mode():
            for _ in range(2):
                sdpa_raw(q,k,v,is_causal=True)
            sync(device)
            with torch.profiler.profile(activities=activities,record_shapes=True) as p:
                sdpa_raw(q,k,v,is_causal=True)
                sync(device)
        names=sorted({event.key for event in p.key_averages()})
        return {'status':'ok','Hkv':Hkv,'device':str(device),
                'operators':[s for s in names if any(t in s.lower() for t in
                    ('attention','softmax','bmm','matmul','repeat'))],
                'note':'名称来自本次调用；含 for_cpu 的算子不证明跑了 CUDA FlashAttention'}
    except (RuntimeError,TypeError,AttributeError) as exc:
        return {'status':'unavailable','reason':str(exc),'Hkv':Hkv}

### Profiler 是证据，但不能把名字读错

`profile_dispatch()` 在正式计时之外单独记录实际算子名。
例如 `aten::_scaled_dot_product_flash_attention_for_cpu` 明确包含 `for_cpu`，
它不证明 CUDA FlashAttention 跑通，更不是 RTX 3060 的速度数据。

Profiler 本身有开销。我们不拿开启 profiler 时的耗时当作正式性能结论，
也不假定一次短输入的分派结果适用于所有长度、精度和头数。

来源：[PyTorch profiler](https://docs.pytorch.org/docs/stable/profiler.html)。

## 10. 显存算式：注意力分数与 KV cache 是两种状态

若 `B=1,Hq=8,L=S=4096`，**单份** FP16 分数矩阵的载荷为：
\[
1\times8\times4096\times4096\times2=256\ \mathrm{MiB}.
\]
它还不是完整峰值：还可能有 softmax 权重、工作区、输出、模型参数等。

如果是单 token decode，`L=1,S=4096`，单份分数仅 **64 KiB**。
此时与长 prefill 的工作规模不同，不能期待同样的加速比例。

持久 KV cache 的载荷则是
\[
2\times\text{层数}\times B\times H_{kv}\times S\times d\times\text{元素字节数}.
\]
FlashAttention 主要改变注意力计算的中间结果/搬运；KV cache 减少历史重算；
GQA 减少存储的 KV 头数。三个技术可以配合，但不能把它们当作同一件事。

In [17]:
print("单份prefill分数 MiB：", 1*8*4096*4096*2/1024**2)
print("单份decode分数 KiB：", 1*8*1*4096*2/1024)
print("单层 GQA KV（Hkv=2,d=64,FP16）MiB：", 2*1*1*2*4096*64*2/1024**2)

单份prefill分数 MiB： 256.0
单份decode分数 KiB： 64.0
单层 GQA KV（Hkv=2,d=64,FP16）MiB： 2.0


## 11. 怎样测量才不会得出错误结论？

本课的计时对象是**固定 Q/K/V 的 attention core 推理前向**。
不含投影、RoPE、KV追加、分词、生成循环、网络、请求排队；不能把结果命名为整模型 tokens/s 或 TTFT。

使用相同输入、相同dtype、相同mask语义。数据和手写基线的mask预先创建；先校验输出，再预热。
每一轮做多次调用，求每次平均耗时，再汇总多轮的中位数和四分位数。
多种方法轮换先后顺序。Profiler、打印、`.item()` 和数据生成不在计时区。

CUDA 调用是异步的；这里选择在测量段前后 `torch.cuda.synchronize()`，用 `perf_counter()` 测
**主机观察到的完成耗时**，包含Python调度与等待，不声称它是纯kernel GPU event时间。
内循环摊薄了同步开销；单token微基准仍可能明显受Python/启动开销影响。

来源：[CUDA synchronize](https://docs.pytorch.org/docs/stable/generated/torch.cuda.synchronize.html)、
[PyTorch SDPA 性能教程](https://docs.pytorch.org/tutorials/intermediate/scaled_dot_product_attention_tutorial.html)。

In [18]:
@dataclass(frozen=True)
class BenchConfig:
    lengths: tuple[int,...]=(64,256)
    kv_heads: tuple[int,...]=(4,2)
    batch_size: int=1
    query_heads: int=4
    head_dim: int=64
    warmup: int=3
    repeats: int=7
    inner: int=10
    max_score_mib: int=128

    def __post_init__(self):
        for name in ('batch_size','query_heads','head_dim','warmup','repeats','inner','max_score_mib'):
            positive_int(getattr(self,name),name)
        if not self.lengths or not self.kv_heads:
            raise ValueError('lengths 和 kv_heads 不能为空')
        for n in self.lengths:
            positive_int(n,'length')
        for h in self.kv_heads:
            positive_int(h,'kv_heads')
            if self.query_heads % h:
                raise ValueError('每个 Hkv 都必须整除 Hq')


def summarize_times(samples_ms: list[float]) -> dict:
    if not samples_ms or any(not math.isfinite(x) or x<=0 for x in samples_ms):
        raise ValueError('需要非空、有限、正的时间样本')
    ordered=sorted(samples_ms)
    def quantile(p):
        pos=(len(ordered)-1)*p
        lo=int(pos); hi=min(lo+1,len(ordered)-1)
        return ordered[lo]+(ordered[hi]-ordered[lo])*(pos-lo)
    return {'median_ms':statistics.median(ordered),'p25_ms':quantile(.25),
            'p75_ms':quantile(.75),'samples_ms':list(samples_ms)}


def timing_trial(fn:Callable, device:torch.device, inner:int) -> float:
    # CUDA 前后同步：包含 Python 调用/启动及完成等待，不是纯 kernel event time。
    sync(device)
    t0=time.perf_counter()
    for _ in range(inner):
        output=fn()
        del output  # 不保留上次输出，防止迭代积累。
    sync(device)
    return (time.perf_counter()-t0)*1000/inner

## 12. 峰值显存：单独测，明确基线

只在 CUDA 上测：先分配输入和mask，预热后同步，记录 `memory_allocated()` 基线，
重置峰值统计，执行一次，等待完成，读取 `max_memory_allocated()`。

\[
\text{本次增量峰值}=\text{峰值allocated}-\text{运行前allocated}.
\]

它包含新输出和临时张量，不包含预先存在的输入/权重；也不是整个进程的显存。
`memory_reserved` 表示缓存分配器保留的量，受之前的实验影响，不把它与 allocated 混为一谈。
计时区不调用 `empty_cache()`；显存测量与计时分开。

**CPU 内存没有在本课测量。** CPU 报告中显存值是 `null`（未测），不是0，不用 `tracemalloc`
推断 PyTorch native allocator 的真实峰值。手算载荷也不冒充实际测量。

来源：[峰值 allocated](https://docs.pytorch.org/docs/stable/generated/torch.cuda.memory.max_memory_allocated.html)、
[CUDA 内存管理](https://docs.pytorch.org/docs/stable/notes/cuda.html#memory-management)。

In [19]:
def measure_memory(fn:Callable,device:torch.device) -> dict:
    """与计时分开。QKV/mask 已分配；增量包含输出与临时张量。

    allocated 是本进程 PyTorch CUDA allocator 管理的 tensor 分配，不含所有
    GPU 内存。reserved 受缓存池历史影响，不解读为本次新需求，不相减比较。
    """
    if device.type!='cuda':
        return {'status':'not_measured_cpu','baseline_allocated_bytes':None,
                'peak_allocated_bytes':None,'incremental_peak_bytes':None,
                'peak_reserved_bytes':None,
                'note':'CPU 未测 PyTorch native 内存；null 不是0'}
    gc.collect(); sync(device)
    baseline=torch.cuda.memory_allocated(device)
    torch.cuda.reset_peak_memory_stats(device)
    output=fn()
    sync(device)
    peak=torch.cuda.max_memory_allocated(device)
    reserved=torch.cuda.max_memory_reserved(device)
    output_bytes=output.numel()*output.element_size()
    del output
    return {'status':'measured_cuda','baseline_allocated_bytes':baseline,
            'peak_allocated_bytes':peak,'incremental_peak_bytes':max(0,peak-baseline),
            'output_payload_bytes':output_bytes,'peak_reserved_bytes':reserved,
            'note':'allocator增量，不是nvidia-smi进程占用；不包含先分配好的QKV/mask'}


def benchmark_case(device,dtype,cfg:BenchConfig,phase:str,S:int,Hkv:int) -> dict:
    L=S if phase=='prefill' else 1
    item={'phase':phase,'B':cfg.batch_size,'Hq':cfg.query_heads,'Hkv':Hkv,
          'L':L,'S':S,'d':cfg.head_dim,'dtype':str(dtype),
          'scope':'attention core forward only; fixed QKV; no projection/RoPE/cache append'}
    # 手写参考在低精度下使用 FP32 分数；这里只估算一份，不是峰值保证。
    compute_bytes=8 if dtype==torch.float64 else 4
    score_bytes=cfg.batch_size*cfg.query_heads*L*S*compute_bytes
    item['one_materialized_score_bytes']=score_bytes
    if score_bytes>cfg.max_score_mib*1024**2:
        return {**item,'status':'skipped_guard','reason':'一份分数矩阵超过保护阈值；未分配输入'}
    q,k,v=make_qkv(cfg.batch_size,cfg.query_heads,Hkv,L,S,cfg.head_dim,device,dtype,
                  seed=2300+S+Hkv)
    mask,causal=attention_plan(L,S,phase,device)
    item['mask_is_causal']=causal
    item['explicit_mask_shape']=None if mask is None else list(mask.shape)
    manual_mask = offset_mask(0,L,device) if causal else mask
    item['manual_preallocated_mask_shape']=None if manual_mask is None else list(manual_mask.shape)
    methods={
        'manual': lambda: manual_core(q,k,v,manual_mask,False),
        'sdpa_math':lambda:sdpa_raw(q,k,v,mask,causal),
        'sdpa_auto':lambda:sdpa_raw(q,k,v,mask,causal),
    }
    if device.type=='cuda':
        methods['sdpa_flash_only']=lambda:sdpa_raw(q,k,v,mask,causal)
    contexts={'manual':'auto','sdpa_math':'math','sdpa_auto':'auto','sdpa_flash_only':'flash'}
    results={};active=[]
    with torch.inference_mode():
        # 固定测试另有FP64真值；基准这里用同 dtype 手写实现核对，记录误差。
        expected=manual_core(q,k,v,manual_mask,False)
        for name,fn in methods.items():
            try:
                with warnings.catch_warnings(record=True) as caught:
                    warnings.simplefilter('always')
                    with backend_context(contexts[name]):
                        actual=fn();sync(device)
                        require_close(actual,expected,dtype)
                        stats=error_stats(actual,expected)
                        del actual
                        for _ in range(cfg.warmup):
                            tmp=fn();del tmp
                        sync(device)
                results[name]={'status':'ok','warnings':[str(w.message) for w in caught],
                               'error_vs_manual':stats,'samples_ms':[]}
                active.append(name)
            except AssertionError as exc:
                results[name]={'status':'correctness_failed','reason':str(exc)}
            except (RuntimeError,TypeError,ValueError,NotImplementedError) as exc:
                results[name]={'status':'unsupported_or_error','reason':str(exc),
                               'warnings':[str(w.message) for w in locals().get('caught',[])]}
        del expected
        # 每轮轮换先后次序，减少固定顺序偏差；校验和 profiler 不在计时区。
        for r in range(cfg.repeats):
            order=active[r%len(active):]+active[:r%len(active)] if active else []
            for name in order:
                with backend_context(contexts[name]):
                    ms=timing_trial(methods[name],device,cfg.inner)
                results[name]['samples_ms'].append(ms)
        for name in active:
            results[name].update(summarize_times(results[name]['samples_ms']))
            with backend_context(contexts[name]):
                results[name]['memory']=measure_memory(methods[name],device)
    return {**item,'status':'completed','methods':results}

In [20]:
def run_benchmarks(device,dtype,cfg:BenchConfig) -> list[dict]:
    rows=[]
    for S in cfg.lengths:
        for Hkv in cfg.kv_heads:
            for phase in ('prefill','decode'):
                rows.append(benchmark_case(device,dtype,cfg,phase,S,Hkv))
    return rows


def print_benchmarks(rows) -> None:
    print('阶段      S   Hkv   手写(ms)  math(ms)  auto(ms)  flash-only(ms)')
    for row in rows:
        methods=row.get('methods',{})
        values=[]
        for name in ('manual','sdpa_math','sdpa_auto','sdpa_flash_only'):
            r=methods.get(name,{})
            values.append(f"{r['median_ms']:.4f}" if r.get('status')=='ok' else 'N/A')
        print(f"{row['phase']:8s} {row['S']:4d} {row['Hkv']:5d} "+'  '.join(values))
    print('只测固定QKV的核心前向；不是整模型token速度。CPU显存字段为null。')

## 13. 自动检查与完整实验

固定测试涵盖 MHA/GQA/MQA 的前向和 Q/K/V 梯度、不同形状的mask、在线归一化、
投影模块参数梯度、dropout模式、因果性、缓存、计时统计和CPU未测项。

这些数学检查不等于“所有GPU内核通过”。后端探测结果单独保存。
下面的测试实现是可读的工具代码；第一次学习可先运行，遇到某个问题时再定位相应检查。

In [21]:
class Checks:
    def __init__(self):
        self.names=[]
    def ok(self,condition,name):
        if not bool(condition):
            raise AssertionError(name)
        self.names.append(name)
    def close(self,a,b,name,rtol=1e-8,atol=1e-10):
        torch.testing.assert_close(a,b,rtol=rtol,atol=atol)
        self.names.append(name)
    def raises(self,fn,name,kind=ValueError):
        try:
            fn()
        except kind:
            self.names.append(name)
        else:
            raise AssertionError(name+'：没有按预期拒绝输入')


def run_checks(device=torch.device('cpu')) -> dict:
    c=Checks()
    for Hkv in (4,2,1):
        for phase,L,S in [('prefill',5,5),('decode',1,7),('chunk',3,7)]:
            q,k,v=make_qkv(2,4,Hkv,L,S,8,device,torch.float64,seed=Hkv+L)
            mask,causal=attention_plan(L,S,phase,device)
            xs=[t.clone().requires_grad_() for t in (q,k,v)]
            ys=[t.clone().requires_grad_() for t in (q,k,v)]
            a=manual_attention(*xs,mask,causal)
            b=sdpa_attention(*ys,mask,causal,backend='math')
            prefix=f'Hkv={Hkv}/{phase}'
            c.close(a,b,prefix+'/前向')
            grad=torch.linspace(-.7,1.3,a.numel(),device=device,dtype=a.dtype).reshape(a.shape)
            ga=torch.autograd.grad((a*grad).sum(),xs)
            gb=torch.autograd.grad((b*grad).sum(),ys)
            for label,g1,g2 in zip(('Q','K','V'),ga,gb):
                c.close(g1,g2,prefix+'/'+label+'梯度')
    c.close(offset_mask(3,2),torch.tensor([[1,1,1,1,0],[1,1,1,1,1]],dtype=torch.bool),
            '偏移矩形mask')
    c.ok(attention_plan(1,9,'decode')==(None,False),'单token decode没有未来key')
    c.raises(lambda:attention_plan(2,9,'decode'),'decode拒绝多个query')
    c.raises(lambda:attention_plan(2,3,'prefill'),'prefill拒绝非方阵')
    c.raises(lambda:offset_mask(-1,2),'拒绝负历史')
    rect=rectangular_demo()
    c.ok(rect['decode_correct']==2.5,'decode读取全部4个key')
    c.ok(rect['wrong_upper_left_causal']==1.0,'复现错误左上因果mask')
    q,k,v=make_qkv(1,4,2,4,4,8,device,torch.float64)
    bad=torch.zeros(4,4,dtype=torch.bool,device=device)
    c.raises(lambda:sdpa_attention(q,k,v,bad),'拒绝全屏蔽行')
    c.raises(lambda:sdpa_attention(q,k,v,bad.float()),'拒绝错误mask类型')
    c.raises(lambda:sdpa_attention(q,k,v,offset_mask(0,4,device),True),'拒绝mask与causal同时指定')
    c.raises(lambda:sdpa_attention(q,k[:,:1].float(),v[:,:1].float()),'拒绝dtype不一致')
    c.raises(lambda:sdpa_attention(q[:,:3],k,v),'拒绝非法头数组合')
    c.raises(lambda:sdpa_attention(q,k,v,dropout_p=-.1),'拒绝负dropout')
    c.raises(lambda:sdpa_attention(q,k,v,dropout_p=float('nan')),'拒绝NaN dropout')
    # 非连续 view 的位置和值必须正确，不靠错误 reshape 偶然过关。
    qs=q.transpose(-1,-2).contiguous().transpose(-1,-2)
    c.ok(not qs.is_contiguous(),'构造非连续Q')
    c.close(manual_attention(qs,k,v,is_causal=True),
            sdpa_attention(qs,k,v,is_causal=True,backend='math'),'非连续输入')
    # 在线累积面对不同分块、局部全屏蔽和大分数。
    q2=torch.tensor([[1.,2.],[-.5,1.],[.2,-1.]],dtype=torch.float64)
    k2=torch.tensor([[1.,0.],[0.,1.],[2.,2.],[-1.,1.],[0.,0.]],dtype=torch.float64)
    v2=torch.arange(15,dtype=torch.float64).reshape(5,3)
    mask=offset_mask(2,3)
    for block in (1,2,3,8):
        for allowed in (None,mask):
            scores=q2@k2.T/math.sqrt(2)
            if allowed is not None:
                scores=scores.masked_fill(~allowed,float('-inf'))
            ref=scores.softmax(-1)@v2
            c.close(streaming_attention(q2,k2,v2,block,allowed),ref,
                    f'在线softmax块={block}/mask={allowed is not None}')
    only_last=torch.zeros(3,5,dtype=torch.bool);only_last[:,-1]=True
    c.close(streaming_attention(q2,k2,v2,2,only_last),v2[-1].expand(3,-1),
            '前几个块全屏蔽仍稳定')
    huge_q=torch.ones(1,1,dtype=torch.float64)
    huge_k=torch.tensor([[1000.],[1001.],[999.]],dtype=torch.float64)
    huge_v=torch.tensor([[1.],[2.],[3.]],dtype=torch.float64)
    c.close(streaming_attention(huge_q,huge_k,huge_v,1),
            (huge_q@huge_k.T).softmax(-1)@huge_v,'大logits稳定')
    c.raises(lambda:streaming_attention(q2,k2,v2,0),'块长必须正数')
    online=online_demo()
    c.ok(abs(online['global_softmax_output']-online['online_output'])<1e-12,'全局摘要正确')
    c.ok(abs(online['global_softmax_output']-online['wrong_average_of_local_outputs'])>4.,
         '局部softmax简单平均反例')
    # 完整投影模块的全部参数梯度也应匹配。
    with torch.random.fork_rng(devices=[] if device.type=='cpu' else [torch.cuda.current_device()]):
        torch.manual_seed(2310)
        model=CoreSwapAttention(dropout=.4).to(device=device,dtype=torch.float64).eval()
        x=torch.randn(2,5,32,device=device,dtype=torch.float64,requires_grad=True)
    a,_=model(x,backend='manual')
    b,_=model(x,backend='math')
    c.close(a,b,'集成模块输出')
    params=[x]+list(model.parameters())
    ga=torch.autograd.grad(a.square().sum(),params)
    gb=torch.autograd.grad(b.square().sum(),params)
    for i,(u,w) in enumerate(zip(ga,gb)):
        c.close(u,w,'集成输入/投影梯度'+str(i))
    with torch.inference_mode():
        first,_=model(x,backend='math');second,_=model(x,backend='math')
        c.close(first,second,'eval显式关闭SDPA dropout')
        prefix,_=model(x[:,:3],backend='math')
        changed=x.detach().clone();changed[:,-1]+=7
        future,_=model(changed,backend='math')
        c.close(first[:,:3],prefix,'完整与前缀一致')
        c.close(first[:,:-1],future[:,:-1],'未来变化不泄漏到前面')
        _,cache=model(x[:,:2],backend='math')
        old=tuple(t.clone() for t in cache)
        _,new_cache=model(x[:,2:3],cache=cache,backend='math')
        for i in (0,1):
            c.close(cache[i],old[i],f'旧缓存{i}不被改写')
        c.ok(new_cache[0].shape==(2,2,3,8),'缓存保持Hkv而非Hq')
    model.train()
    c.raises(lambda:model(x[:,:1],cache=cache,backend='math'),'训练时拒绝缓存')
    # no_grad/eval 控制不同；以确定性的 monkeypatch 核对传入参数。
    from unittest.mock import patch
    recorded=[]
    real_sdpa=F.scaled_dot_product_attention
    def capture(*args,**kwargs):
        recorded.append(kwargs.get('dropout_p'))
        return real_sdpa(*args,**{**kwargs,'dropout_p':0.0})
    with patch.object(F,'scaled_dot_product_attention',side_effect=capture):
        model.train();model(x,backend='math')
        model.eval();model(x,backend='math')
    c.ok(recorded==[.4,0.0],'train/eval正确传递dropout_p')
    r=integration_demo(device)
    c.ok(r['weights_unchanged'],'更换核心没有修改参数')
    c.ok(r['full_vs_chunks_max_abs']<1e-9,'分块缓存整体验证')
    c.ok(r['K_max_abs']<1e-9 and r['V_max_abs']<1e-9,'完整与缓存K/V一致')
    # 测量工具自身的检查，不把错误后端当作通过。
    summary=summarize_times([3.,1.,2.,4.,5.])
    c.ok(summary['median_ms']==3.,'中位数')
    c.ok(summary['p25_ms']==2. and summary['p75_ms']==4.,'四分位数')
    c.raises(lambda:summarize_times([]),'拒绝空计时结果')
    c.raises(lambda:summarize_times([float('nan')]),'拒绝NaN计时')
    c.raises(lambda:BenchConfig(kv_heads=(3,)),'拒绝不可整除基准配置')
    c.raises(lambda:BenchConfig(inner=0),'inner必须正数')
    # 用虚拟时钟验证除以inner和调用次数，不靠运行速度碰运气。
    calls=[]
    def counted():
        calls.append(1)
        return torch.zeros(1)
    with patch.object(time,'perf_counter',side_effect=[10.0,10.012]):
        fake_ms=timing_trial(counted,torch.device('cpu'),3)
    c.ok(len(calls)==3,'计时实际调用inner次')
    c.ok(abs(fake_ms-4.0)<1e-8,'计时换算毫秒并除以inner')
    cpu_memory=measure_memory(lambda:torch.zeros(1),torch.device('cpu'))
    c.ok(cpu_memory['incremental_peak_bytes'] is None,'CPU显存是null不是0')
    skip=probe_one(q.cpu(),k.cpu(),v.cpu(),None,True,'flash')
    c.ok(skip['status']=='skipped','CPU不冒称CUDA Flash测试通过')
    serial=json.dumps({'summary':summary,'memory':cpu_memory},allow_nan=False)
    c.ok('null' in serial,'JSON显式保留未测项')
    return {'passed':len(c.names),'names':c.names,'scope':'数学与集成测试；不等于所有GPU后端通过'}

In [22]:
CHECK_REPORT = run_checks(DEVICE)
print("固定检查通过：", CHECK_REPORT["passed"])
print("最近的检查：")
for name in CHECK_REPORT["names"][-8:]:
    print(" -",name)

固定检查通过： 93
最近的检查：
 - 拒绝NaN计时
 - 拒绝不可整除基准配置
 - inner必须正数
 - 计时实际调用inner次
 - 计时换算毫秒并除以inner
 - CPU显存是null不是0
 - CPU不冒称CUDA Flash测试通过
 - JSON显式保留未测项


In [23]:
def run_experiment(device:torch.device,dtype:torch.dtype,output_dir:Path,
                   cfg:BenchConfig=BenchConfig(),*,profile=True,checks=None) -> dict:
    output_dir=Path(output_dir)
    output_dir.mkdir(parents=True,exist_ok=True)
    report={
        'lesson':23,'environment':environment(device,dtype),'benchmark_config':asdict(cfg),
        'checks':checks if checks is not None else run_checks(device),
        'online_softmax':online_demo(),'rectangular_mask':rectangular_demo(),
        'integration':integration_demo(device),
        'backend_probes':backend_probes(device,dtype),
        'profiler':([profile_dispatch(device,dtype,h) for h in (4,2)] if profile else
                    [{'status':'disabled','reason':'用户关闭 profiler'}]),
        'benchmarks':run_benchmarks(device,dtype,cfg),
        'sources':SOURCES,
        'limits':[
            '仅 attention core 推理前向基准；没有测整模型、服务TTFT或tokens/s。',
            '计时为同步wall-clock分组平均；不是逐个kernel的GPU event时间。',
            'CPU只测延迟；CUDA不可用时不提供GPU内核和显存实测结论。',
            '默认不安装第三方flash-attn；SDPA后端由本机输入/版本/设备决定。',
            '固定随机QKV，无训练或质量评测；不加载前课检查点。',
            '自动分派后端可能随shape、dtype、mask、GQA改变。',
            '手写低精度参考显式使用FP32中间值，额外转换计入其耗时。',
            '无padding的固定形状输入；无编译、量化、分页缓存和并发服务。',
            '返回完整注意力权重会重新引入大矩阵；本课优化路径只返回output。',
        ]}
    def write(name,data):
        (output_dir/name).write_text(json.dumps(data,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')
    write('experiment.json',report)
    write('backend_report.json',{'environment':report['environment'],
          'probes':report['backend_probes'],'profiler':report['profiler']})
    write('benchmark.json',{'config':report['benchmark_config'],
          'environment':report['environment'],'results':report['benchmarks'],'limits':report['limits']})
    return report

### 执行设置

默认只测试长度64、256，batch=1，Hq=4，Hkv分别为4和2，每头64维。
3060 环境跑通后，可以先改成 `lengths=(128,512,1024)`，再按实际显存逐步增加。
不要一次把batch、长度和精度全部改掉，否则难以归因。

`max_score_mib` 只是**单份手写分数载荷**的保护阈值，不是显存上限保证。
报告会覆盖输出目录里的同名JSON；不会写入或修改前几课检查点。

In [24]:
BENCH = BenchConfig(lengths=(64,256), kv_heads=(4,2),
                    warmup=3, repeats=7, inner=10)
REPORT = run_experiment(DEVICE, DTYPE, OUTPUT_DIR, BENCH,
                        profile=True, checks=CHECK_REPORT)
print_benchmarks(REPORT["benchmarks"])
print("报告目录：", OUTPUT_DIR.resolve())

/opt/pyvenv/lib/python3.13/site-packages/torch/profiler/profiler.py:217: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


阶段      S   Hkv   手写(ms)  math(ms)  auto(ms)  flash-only(ms)
prefill    64     4 0.1168  0.1668  0.0809  N/A
decode     64     4 0.0211  0.0211  0.0089  N/A
prefill    64     2 0.1303  0.1707  0.0769  N/A
decode     64     2 0.0302  0.0341  0.0092  N/A
prefill   256     4 1.2668  1.5164  0.8995  N/A
decode    256     4 0.0326  0.0400  0.0192  N/A
prefill   256     2 1.2894  1.4906  0.9149  N/A
decode    256     2 0.0820  0.0587  0.0180  N/A
只测固定QKV的核心前向；不是整模型token速度。CPU显存字段为null。
报告目录： /mnt/data/_lesson23_work/notebook_run/lesson_23_outputs


In [25]:
from collections import Counter
print("后端探测状态：",dict(Counter(r["status"] for r in REPORT["backend_probes"])))
for item in REPORT["profiler"]:
    print(json.dumps(item,ensure_ascii=False,indent=2))

# 查看一份实际内存记录：CPU时是null，不会伪造显存数字。
first_case = REPORT["benchmarks"][0]
for name, result in first_case.get("methods",{}).items():
    print(name, result["status"], result.get("memory"))

后端探测状态： {'ok': 18, 'skipped': 27}
{
  "status": "ok",
  "Hkv": 4,
  "device": "cpu",
  "operators": [
    "aten::_scaled_dot_product_flash_attention_for_cpu",
    "aten::scaled_dot_product_attention"
  ],
  "note": "名称来自本次调用；含 for_cpu 的算子不证明跑了 CUDA FlashAttention"
}
{
  "status": "ok",
  "Hkv": 2,
  "device": "cpu",
  "operators": [
    "aten::_scaled_dot_product_flash_attention_for_cpu",
    "aten::scaled_dot_product_attention"
  ],
  "note": "名称来自本次调用；含 for_cpu 的算子不证明跑了 CUDA FlashAttention"
}
manual ok {'status': 'not_measured_cpu', 'baseline_allocated_bytes': None, 'peak_allocated_bytes': None, 'incremental_peak_bytes': None, 'peak_reserved_bytes': None, 'note': 'CPU 未测 PyTorch native 内存；null 不是0'}
sdpa_math ok {'status': 'not_measured_cpu', 'baseline_allocated_bytes': None, 'peak_allocated_bytes': None, 'incremental_peak_bytes': None, 'peak_reserved_bytes': None, 'note': 'CPU 未测 PyTorch native 内存；null 不是0'}
sdpa_auto ok {'status': 'not_measured_cpu', 'baseline_allocated_bytes': Non

### 怎样读刚生成的表格？

同一行比较相同输入的不同实现；不把prefill和decode当成相同工作量。
跨Hkv比较时，结构已经改变，不能把某个较小参数值的耗时当作同一数学问题的内核加速。

本次发布验证只在CPU运行；即使auto比math快，也不能称为“3060上的FlashAttention加速”。
运行太短时，调度、频率和系统负载会影响结果；先看多轮原始样本和四分位数，而不是只截取最小值。
如果把一个精度改成FP16，另一个保持FP32，得到的是精度与实现一起改变的结果。

JSON 文件：
- `experiment.json`：环境、数学实验、集成、全部探测和基准。
- `backend_report.json`：逐项后端状态、错误原因与profiler算子名。
- `benchmark.json`：计时条件、原始样本、误差、显存与限制。

不同Notebook/脚本运行的计时应当有波动，不要求耗时逐位一致。

## 14. 独立练习 A：写缓存场景的 SDPA 调用

实现 `cached_sdpa_ex(q,k,v,past_len)`，要求：

输入只包含本次新Q；K/V已包含历史与新增，必须满足 `S=past_len+L`。
没有padding，所有历史有效。历史为0时用方阵因果；历史不为0且L=1时允许全部key；
其他情况显式构造偏移mask。支持GQA；推理dropout为0。

可复用输入校验，但请亲手调用 `F.scaled_dot_product_attention`，不要直接返回现成wrapper。

In [26]:
def cached_sdpa_ex(q, k, v, past_len):
    raise NotImplementedError("补全prefill/decode/chunk的不同语义")

RUN_EXERCISE_A = False

In [27]:
def check_exercise_a(fn):
    count=0
    for past,L,Hkv in [(0,4,4),(3,1,2),(3,2,1)]:
        q,k,v=make_qkv(1,4,Hkv,L,past+L,8,torch.device("cpu"),torch.float64)
        xs=[t.clone().requires_grad_() for t in (q,k,v)]
        ys=[t.clone().requires_grad_() for t in (q,k,v)]
        actual=fn(*xs,past)
        expected=manual_attention(*ys,mask=offset_mask(past,L))
        torch.testing.assert_close(actual,expected,rtol=1e-9,atol=1e-10);count+=1
        ga=torch.autograd.grad(actual.square().sum(),xs)
        gb=torch.autograd.grad(expected.square().sum(),ys)
        for a,b in zip(ga,gb):
            torch.testing.assert_close(a,b,rtol=1e-8,atol=1e-10);count+=1
    try:
        fn(q,k,v,99)
    except ValueError:
        count+=1
    else:
        raise AssertionError("没有拒绝错误历史长度")
    print("练习A通过：",count,"项")
    return count

if RUN_EXERCISE_A:
    check_exercise_a(cached_sdpa_ex)
else:
    print("练习A尚未开启；请完成后将开关改成True")

练习A尚未开启；请完成后将开关改成True


## 15. 独立练习 B：合并在线 softmax 摘要

实现 `merge_summary_ex(m,z,u,block_scores,block_values)`。
输入形状与正文相同，返回更新后的 `(m,z,u)`；不要调用已有 `merge_summary()`。

重点：新最大值改变时，对旧分母和旧分子乘同一个缩放因子；当前块可能全被屏蔽。
不要对块输出单独归一化后简单平均。允许最终尚未看到任何有效key时 `z=0`，
不要在合并摘要阶段提前执行 `u/z`。

In [28]:
def merge_summary_ex(m, z, u, block_scores, block_values):
    raise NotImplementedError("根据公式更新最大值、分母与加权分子")

RUN_EXERCISE_B = False

In [29]:
def check_exercise_b(fn):
    scores=torch.tensor([[0.,0.,3.,3.]],dtype=torch.float64,requires_grad=True)
    values=torch.tensor([[0.],[0.],[10.],[10.]],dtype=torch.float64,requires_grad=True)
    m=torch.full((1,1),-torch.inf,dtype=torch.float64)
    z=torch.zeros(1,1,dtype=torch.float64);u=torch.zeros(1,1,dtype=torch.float64)
    for start in (0,2):
        m,z,u=fn(m,z,u,scores[:,start:start+2],values[start:start+2])
    actual=u/z
    expected=scores.softmax(-1)@values
    torch.testing.assert_close(actual,expected,rtol=1e-10,atol=1e-10)
    ga=torch.autograd.grad(actual.sum(),(scores,values),retain_graph=True)
    gb=torch.autograd.grad(expected.sum(),(scores,values))
    for a,b in zip(ga,gb):
        torch.testing.assert_close(a,b,rtol=1e-9,atol=1e-10)
    # 第一块完全屏蔽，不允许产生NaN。
    m0=torch.full((1,1),-torch.inf,dtype=torch.float64)
    z0=torch.zeros(1,1,dtype=torch.float64);u0=torch.zeros(1,1,dtype=torch.float64)
    m1,z1,u1=fn(m0,z0,u0,torch.full((1,2),-torch.inf,dtype=torch.float64),values[:2].detach())
    if torch.isnan(m1).any() or not torch.isfinite(z1).all() or not torch.isfinite(u1).all():
        raise AssertionError("全屏蔽块导致NaN/Inf")
    m2,z2,u2=fn(m1,z1,u1,scores[:,2:].detach(),values[2:].detach())
    torch.testing.assert_close(u2/z2,torch.tensor([[10.]],dtype=torch.float64))
    print("练习B通过：5项（包括梯度和全屏蔽块）")
    return 5

if RUN_EXERCISE_B:
    check_exercise_b(merge_summary_ex)
else:
    print("练习B尚未开启")

练习B尚未开启


## 16. 独立练习 C：写同步计时函数

实现 `measure_ex(fn, device, warmup, repeats, inner)`，返回 `summarize_times(samples_ms)`。
预热不计时；预热后同步；每轮开始和结束都调用 `sync(device)`；每轮执行 `inner` 次，
换成毫秒后除以inner。整个过程使用 `torch.inference_mode()`，不保留所有输出。

下面的检查会临时替换时钟和同步函数，验证调用次数、单位换算以及是否误把多次调用当作一次。
这是测试工具，不会真的等待模拟时钟显示的时间，也不等于测出了某个模型的性能。

In [30]:
def measure_ex(fn, device, warmup=2, repeats=3, inner=3):
    raise NotImplementedError("预热、同步、计时、换算、统计")

RUN_EXERCISE_C = False

In [31]:
def check_exercise_c(fn):
    from types import SimpleNamespace
    from unittest.mock import patch
    stamps=iter([10.,10.009,20.,20.012,30.,30.015])
    calls=[];sync_calls=[]
    def operation():
        calls.append(torch.is_grad_enabled())
        return torch.zeros(1)
    fake_time=SimpleNamespace(perf_counter=lambda:next(stamps))
    fake_sync=lambda device:sync_calls.append(str(device))
    with patch.dict(fn.__globals__, {"time":fake_time,"sync":fake_sync}):
        result=fn(operation,torch.device("cpu"),warmup=2,repeats=3,inner=3)
    if len(calls)!=11:
        raise AssertionError("调用次数应为 warmup + repeats*inner = 11")
    if any(calls):
        raise AssertionError("推理计时没有关闭梯度记录")
    if len(sync_calls)!=7:
        raise AssertionError("应在预热后同步一次，每个计时段前后各同步一次")
    torch.testing.assert_close(torch.tensor(result["samples_ms"]),torch.tensor([3.,4.,5.]),rtol=1e-5,atol=1e-5)
    if abs(result["median_ms"]-4)>1e-7:
        raise AssertionError("中位数或毫秒/inner换算错误")
    print("练习C通过：5项（模拟时钟验证，不是性能数据）")
    return 5

if RUN_EXERCISE_C:
    check_exercise_c(measure_ex)
else:
    print("练习C尚未开启")

练习C尚未开启


---
## 17. 参考答案：先独立完成，再往下阅读

参考函数使用不同名字，不覆盖你的实现。检查通过只说明这些明确的案例成立；
修改输入契约后仍需要补测试。以下答案会实际运行验证，不能把它们的通过当作你的练习通过。

In [32]:
def cached_sdpa_answer(q,k,v,past_len):
    validate_qkv(q,k,v)
    if type(past_len) is not int or past_len<0 or k.shape[-2]!=past_len+q.shape[-2]:
        raise ValueError("历史长度与Q/K长度不一致")
    L=q.shape[-2]
    if past_len==0:
        mask,causal=None,True
    elif L==1:
        mask,causal=None,False
    else:
        queries=past_len+torch.arange(L,device=q.device)
        keys=torch.arange(past_len+L,device=q.device)
        mask=keys[None,:]<=queries[:,None]
        causal=False
    return F.scaled_dot_product_attention(q,k,v,attn_mask=mask,is_causal=causal,
                                         dropout_p=0.0,enable_gqa=q.shape[1]!=k.shape[1])

ANSWER_A_CHECKS=check_exercise_a(cached_sdpa_answer)

练习A通过： 13 项


In [33]:
def merge_summary_answer(m,z,u,block_scores,block_values):
    new_max=torch.maximum(m,block_scores.amax(-1,keepdim=True))
    safe=torch.where(torch.isfinite(new_max),new_max,torch.zeros_like(new_max))
    old_scale=(m-safe).exp()
    block_exp=(block_scores-safe).exp()
    new_z=old_scale*z+block_exp.sum(-1,keepdim=True)
    new_u=old_scale*u+block_exp@block_values
    return new_max,new_z,new_u

ANSWER_B_CHECKS=check_exercise_b(merge_summary_answer)

练习B通过：5项（包括梯度和全屏蔽块）


In [34]:
def measure_answer(fn,device,warmup=2,repeats=3,inner=3):
    samples=[]
    with torch.inference_mode():
        for _ in range(warmup):
            output=fn()
            del output
        sync(device)
        for _ in range(repeats):
            sync(device)
            start=time.perf_counter()
            for _ in range(inner):
                output=fn()
                del output
            sync(device)
            elapsed=(time.perf_counter()-start)*1000/inner
            samples.append(elapsed)
    return summarize_times(samples)

ANSWER_C_CHECKS=check_exercise_c(measure_answer)

练习C通过：5项（模拟时钟验证，不是性能数据）


## 18. 脚本使用与本课验收

```bash
# 固定数学/集成检查，不写报告
python lesson_23_sdpa_flashattention_and_benchmarking.py --checks-only

# CPU 默认实验
python lesson_23_sdpa_flashattention_and_benchmarking.py

# 本机已有可用CUDA：明确请求GPU，不可用就报错而不是悄悄CPU测试
python lesson_23_sdpa_flashattention_and_benchmarking.py --device cuda --dtype float16 --lengths 128,512,1024

# profiler环境不可用时，只跳过profiler，不改变其他实验
python lesson_23_sdpa_flashattention_and_benchmarking.py --skip-profile
```

也支持 `--device auto`、`--kv-heads 4,2,1`、`--repeats 11`、`--inner 20`、`--output-dir my_run_23`。
每次实验保存自己的配置，不把改变条件后的数值混在一起。

**完成标准：** 能解释局部softmax反例；能写对三种阶段的mask；能读懂某个后端失败原因；
能区分单份载荷、增量allocated峰值与reserved；能解释为什么CPU的“flash”算子不是3060实测。

**下一课：第24课——FP32、FP16、BF16与混合精度。**
从本课的dtype选择继续，理解数值范围、舍入、autocast和梯度缩放，分别检查精度、稳定性与资源开销。

## 参考资料与结论边界

1. [PyTorch SDPA](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html)：mask、dropout、GQA和数值约定。
2. [后端选择](https://docs.pytorch.org/docs/stable/generated/torch.nn.attention.sdpa_kernel.html)：强制后端与上下文恢复。
3. [FlashAttention 原论文](https://arxiv.org/abs/2205.14135)：IO-aware tiling、全局归一化与反向重计算。
4. [官方性能教程](https://docs.pytorch.org/tutorials/intermediate/scaled_dot_product_attention_tutorial.html)：不同实现的测量和限制。
5. [PyTorch profiler](https://docs.pytorch.org/docs/stable/profiler.html)：记录真实执行算子。
6. [CUDA 同步](https://docs.pytorch.org/docs/stable/generated/torch.cuda.synchronize.html)、[显存峰值](https://docs.pytorch.org/docs/stable/generated/torch.cuda.memory.max_memory_allocated.html)：测量口径。
7. [浮点精度说明](https://docs.pytorch.org/docs/stable/notes/numerical_accuracy.html)：为什么数学等价不要求所有输出逐位一致。

文档会更新，请以脚本保存的本机PyTorch版本和实际探测结果解释报告。
本课没有给出不同FlashAttention版本的最新排名、没有测整模型或服务，也没有在CPU验证中声称测过GPU。